# PCA Sensitivity - Harmonization Trade-off

## Objective
Visualize the trade-off between **site-invariance** (privacy) and **pathology classification** (utility) across all harmonization methods and PCA variance levels.

Each point is one hospital. Axes show the delta relative to the raw baseline:
- **x (ΔSite MCC):** negative = harmonization *reduced* site discriminability → better privacy
- **y (ΔPath AUC):** positive = harmonization *improved* pathology AUC → better utility

The ideal outcome (top-left quadrant) means a method simultaneously reduces site effects and maintains or improves pathology performance.

## Key Insight
Without PCA, harmonized methods (especially ComBat) land in the **bottom-right** quadrant — worsening both privacy and utility (the paradox). With PCA, points shift toward top-left.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name != 'eeg-site-effects':
    os.chdir('../..')
print('Working directory:', Path.cwd())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_PATH_SITE      = 'results/tables/05_pca_sensitivity/pca_sensitivity_results_site.csv'
RESULTS_FULL_PATH_SITE = 'results/tables/05_pca_sensitivity/pca_sensitivity_results_site_full.csv'
RESULTS_PATH_PATHO     = 'results/tables/05_pca_sensitivity/pca_sensitivity_results_patho_full_single_catboost.csv'
FIGURES_DIR            = 'results/figures/05_pca_sensitivity/tradeoff'

METHODS    = ['sitewise', 'combat', 'neurocombat', 'covbat']
PCA_ORDER  = ['None', '1.0', '0.99', '0.95', '0.90', '0.80']

## Build delta_df

In [ ]:
# --- Site data (wide format: one column per hospital) ---
site_df      = pd.read_csv(RESULTS_PATH_SITE)
site_full_df = pd.read_csv(RESULTS_FULL_PATH_SITE)
site_df['pca_var']      = site_df['pca_var'].fillna(0.0)   # NaN = no PCA
site_full_df['pca_var'] = site_full_df['pca_var'].fillna(1.0)  # NaN = full PCA
site_all = pd.concat([site_df, site_full_df], ignore_index=True)

# Melt to long format
site_cols = [c for c in site_all.columns if c.startswith('mcc_') and c != 'mcc_overall']
site_long = site_all.melt(
    id_vars=['method', 'pca_var', 'fold_id'],
    value_vars=site_cols,
    var_name='hospital',
    value_name='site_mcc'
)
site_long['hospital'] = site_long['hospital'].str.removeprefix('mcc_')
site_mean = site_long.groupby(['method', 'pca_var', 'hospital'])['site_mcc'].mean().reset_index()

# --- Patho data (long format) ---
patho_df = pd.read_csv(RESULTS_PATH_PATHO)
pca_str_to_float = {'none': 0.0, 'all': 1.0, '0.99': 0.99, '0.95': 0.95, '0.9': 0.90, '0.8': 0.80}
patho_df['pca_var'] = patho_df['pca_var'].map(pca_str_to_float)
patho_mean = (
    patho_df
    .groupby(['method', 'pca_var', 'hospital'])['auc']
    .mean()
    .reset_index()
    .rename(columns={'auc': 'patho_auc'})
)

# --- Merge and compute deltas vs raw ---
merged = site_mean.merge(patho_mean, on=['method', 'pca_var', 'hospital'])

raw_ref = (
    merged[merged['method'] == 'raw']
    [['pca_var', 'hospital', 'site_mcc', 'patho_auc']]
    .rename(columns={'site_mcc': 'raw_site_mcc', 'patho_auc': 'raw_patho_auc'})
)

delta_df = (
    merged[merged['method'].isin(METHODS)]
    .merge(raw_ref, on=['pca_var', 'hospital'])
)
delta_df['delta_mcc'] = delta_df['site_mcc'] - delta_df['raw_site_mcc']
delta_df['delta_auc'] = delta_df['patho_auc'] - delta_df['raw_patho_auc']

# Map pca_var floats to display labels
pca_label_map = {0.0: 'None', 1.0: '1.0', 0.99: '0.99', 0.95: '0.95', 0.90: '0.90', 0.80: '0.80'}
delta_df['pca_var'] = delta_df['pca_var'].map(pca_label_map)

print(delta_df.shape)
delta_df.head()

## Trade-off grid: rows = method, cols = PCA variance

In [ ]:
sns.set_style('whitegrid')

g = sns.FacetGrid(
    delta_df,
    row='method',
    col='pca_var',
    row_order=METHODS,
    col_order=PCA_ORDER,
    margin_titles=True,
    height=2.5,
    aspect=1.2,
    sharex=True,
    sharey=True,
)

g.map(sns.scatterplot, 'delta_mcc', 'delta_auc', alpha=0.6, s=30, color='#3366cc')
g.map(plt.axvline, x=0, color='black', linestyle='-', alpha=0.3)
g.map(plt.axhline, y=0, color='black', linestyle='-', alpha=0.3)
g.map(sns.regplot, 'delta_mcc', 'delta_auc',
      scatter=False, color='red', ci=None, line_kws={'linewidth': 1})

g.set_axis_labels(
    r'$\Delta$ Site MCC  (← better privacy)',
    r'$\Delta$ Path AUC  (↑ better utility)'
)
g.fig.suptitle(
    'Harmonization trade-off: rows = method, cols = PCA variance',
    fontsize=14, y=1.01
)

os.makedirs(FIGURES_DIR, exist_ok=True)
g.fig.savefig(f'{FIGURES_DIR}/tradeoff_grid.png', dpi=150, bbox_inches='tight')
plt.show()